implementation of a CNN model using `pytorch` and evalutaion with CIFAR10 data set. 

In [ ]:
import time
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import Subset

# Set the device to GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Data Preparation: Define transformations and load CIFAR10 dataset
# Transformations for training data (includes data augmentation)
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(), # Randomly flip images horizontally
    transforms.RandomCrop(32, padding=4), # Randomly crop images
    transforms.ToTensor(), # Convert images to PyTorch tensors
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)) # Normalize image pixel values
])

# Transformations for test data (onlyToTensor and Normalize)
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

batch_size = 64

# Load CIFAR10 training and test datasets
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Option to use a smaller subset of the data for faster iteration, especially on CPU
USE_SUBSET = True

if USE_SUBSET:
    train_indices = list(range(10000)) # Use first 10,000 training samples
    test_indices = list(range(2000)) # Use first 2,000 test samples
    trainset = Subset(trainset, train_indices)
    testset = Subset(testset, test_indices)

# Create data loaders for batching and shuffling
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=0)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=0)

# Define the class names for CIFAR10 dataset
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# Model Definition: A simple Convolutional Neural Network (CNN)
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Feature extraction layers (Convolutional and Pooling layers)
        self.features = nn.Sequential(
            # First convolutional block
            nn.Conv2d(3, 16, kernel_size=5, padding=2), # 3 input channels (RGB), 16 output channels
            nn.ReLU(), # Activation function
            nn.MaxPool2d(2), # Max pooling to reduce spatial dimensions

            # Second convolutional block
            nn.Conv2d(16, 32, kernel_size=5, padding=2), # 16 input channels, 32 output channels
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # Classifier layers (Fully Connected layers)
        self.classifier = nn.Sequential(
            nn.Flatten(), # Flatten the output of convolutional layers into a 1D vector
            nn.Linear(32 * 8 * 8, 128), # First fully connected layer
            nn.ReLU(),
            nn.Dropout(0.3), # Dropout for regularization to prevent overfitting
            nn.Linear(128, 10) # Output layer with 10 classes (for CIFAR10)
        )

    # Forward pass definition
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Instantiate the model and move it to the configured device (CPU/GPU)
model = CNNModel().to(device)
print(model)

# Define the loss function (Cross Entropy for classification)
criterion = nn.CrossEntropyLoss()
# Define the optimizer (Adam optimizer)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training and Evaluation Functions
# Function to evaluate the model's performance on a given dataset
def evaluate(model, loader):
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0
    loss_total = 0.0

    with torch.no_grad(): # Disable gradient calculation for inference
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            _, predicted = torch.max(outputs, 1) # Get the predicted class
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            loss_total += loss.item() * labels.size(0)

    return loss_total / total, correct / total

# Training loop parameters
num_epochs = 5
train_losses = []
test_accs = []

start_time = time.time()

# Main training loop
for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    total_seen = 0

    for batch_idx, (images, labels) in enumerate(trainloader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad() # Zero the gradients before backpropagation
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward() # Perform backpropagation to compute gradients
        optimizer.step() # Update model parameters

        running_loss += loss.item() * labels.size(0)
        total_seen += labels.size(0)

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, batch {batch_idx}/{len(trainloader)}, loss={loss.item():.4f}")

    # Calculate and record training and testing metrics for the epoch
    train_loss = running_loss / total_seen
    test_loss, test_acc = evaluate(model, testloader)

    train_losses.append(train_loss)
    test_accs.append(test_acc)

    print(f"Epoch {epoch+1:02d}/{num_epochs} | train_loss={train_loss:.4f} | test_loss={test_loss:.4f} | test_acc={test_acc:.4f}")

end_time = time.time()

print(f"\nTraining time: {end_time - start_time:.2f} sec")
print(f"Best test accuracy: {max(test_accs):.4f}")

# Plotting Training Results
# Plotting training loss over epochs
plt.figure(figsize=(6,4))
plt.plot(train_losses)
plt.title("Q3 training loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.grid(True)
plt.savefig("q3_training_loss.png", dpi=200, bbox_inches="tight")
plt.close()
print("Saved plot: q3_training_loss.png")

# Plotting test accuracy over epochs
plt.figure(figsize=(6,4))
plt.plot(test_accs)
plt.title("Q3 test accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.grid(True)
plt.savefig("q3_test_accuracy.png", dpi=200, bbox_inches="tight")
plt.close()
print("Saved plot: q3_test_accuracy.png")

# Per-class Accuracy Calculation
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

model.eval() # Set model to evaluation mode
with torch.no_grad(): # Disable gradient calculation
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1) # Get predicted class for each image

        # Accumulate correct and total predictions for each class
        for label, prediction in zip(labels, predictions):
            label_name = classes[label.item()]
            if label == prediction:
                correct_pred[label_name] += 1
            total_pred[label_name] += 1

print("\nPer-class accuracy:")
# Print accuracy for each class
for classname in classes:
    if total_pred[classname] > 0:
        acc = 100 * correct_pred[classname] / total_pred[classname]
        print(f"{classname:5s}: {acc:.2f}%")

Using device: cpu


100%|██████████| 170M/170M [00:03<00:00, 49.0MB/s]


CNNModel(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)
Epoch 1/5, batch 0/157, loss=2.2957
Epoch 1/5, batch 100/157, loss=1.9421
Epoch 01/5 | train_loss=2.0003 | test_loss=1.7291 | test_acc=0.3730
Epoch 2/5, batch 0/157, loss=1.8466
Epoch 2/5, batch 100/157, loss=1.6988
Epoch 02/5 | train_loss=1.7147 | test_loss=1.4874 | test_acc=0.4770
Epoch 3/5, batch 0/157, loss=1.6549
Epoch 3/5, batch 100/157, loss=1.606

As we can see in these results, we can see, there is a a test accuracy of 50% in the model. This iss can be expected since the number of samples and resources given are not the optimal. Witha  bigger dataset and more traning and more epochs this result could get better. This would need more resources since traning it is more computationaly heavy.